In [118]:
import sys
sys.path.append("..")
import gymnasium as gym
from gymnasium.wrappers import RecordVideo

env = gym.make("HalfCheetah-v5")

In [119]:
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]
state_dim, action_dim

(17, 6)

In [120]:
import numpy as np
import torch
from src.fnn import FNN
from src.replay_buffer import ReplayBuffer
from src.utils import device, polyak_update
from src.variable_fnn import VariableFNN
from torch import nn, optim

rng = np.random.default_rng(0)
num_experts = 4

actor = FNN(
    input_size = state_dim,
    hidden_size = 256,
    num_hidden_layers = 5,
    output_size = num_experts * 2 * action_dim + num_experts,
).to(device)

critic = FNN(
    input_size = state_dim + action_dim,
    hidden_size = 256,
    num_hidden_layers = 5,
    output_size = 1,
).to(device)

critic1 = FNN(
    input_size = state_dim + action_dim,
    hidden_size = 256,
    num_hidden_layers = 5,
    output_size = 1,
).to(device)
critic2 = FNN(
    input_size = state_dim + action_dim,
    hidden_size = 256,
    num_hidden_layers = 5,
    output_size = 1,
).to(device)

target_actor = FNN(
    input_size = state_dim,
    hidden_size = 256,
    num_hidden_layers = 5,
    output_size = num_experts * 2 * action_dim + num_experts,
).to(device)

target_critic = FNN(
    input_size = state_dim + action_dim,
    hidden_size = 256,
    num_hidden_layers = 5,
    output_size = 1,
).to(device)

target_critic1 = FNN(
    input_size = state_dim + action_dim,
    hidden_size = 256,
    num_hidden_layers = 5,
    output_size = 1,
).to(device)
target_critic2 = FNN(
    input_size = state_dim + action_dim,
    hidden_size = 256,
    num_hidden_layers = 5,
    output_size = 1,
).to(device)

target_actor.load_state_dict(actor.state_dict())
for param in target_actor.parameters():
    param.requires_grad_(False)
target_critic.load_state_dict(critic.state_dict())
for param in target_critic.parameters():
    param.requires_grad_(False)
target_critic1.load_state_dict(critic1.state_dict())
for param in target_critic1.parameters():
    param.requires_grad_(False)
target_critic2.load_state_dict(critic2.state_dict())
for param in target_critic2.parameters():
    param.requires_grad_(False)

In [121]:
from torch.distributions import Distribution, Normal, Categorical
from torch import Tensor
import torch
from torch.nn.functional import softplus

max_kl = 10

class DiagonalGaussianMixture(Distribution):
    arg_constraints = {}
    
    def __init__(self, mus: Tensor, sigmas: Tensor, logits: Tensor):
        super().__init__()
        assert len(mus.shape) == 3 and mus.shape == sigmas.shape and mus.shape[: -1] == logits.shape
        self._mus = mus
        self._sigmas = sigmas
        self._experts = []
        for i in range(logits.shape[1]):
            mu, sigma = mus[:, i], sigmas[:, i]
            expert = Normal(loc = mu, scale = sigma)
            self._experts.append(expert)
        self._selector = Categorical(logits = logits)

    @staticmethod
    def from_params(params: Tensor, num_experts: int, sample_dim: int) -> "DiagonalGaussianMixture":
        vec_length = num_experts * sample_dim
        mus = params[:, : vec_length].view(-1, num_experts, sample_dim)
        sigmas = softplus(params[:, vec_length : 2 * vec_length].view(-1, num_experts, sample_dim))
        logits = params[:, -num_experts :]
        return DiagonalGaussianMixture(mus, sigmas, logits)

    def log_prob(self, sample) -> Tensor:
        assert len(sample.shape) == 2
        log_probs = []
        for expert_idx, expert in enumerate(self._experts):
            expert_log_prob = self._selector.log_prob(
                torch.full((sample.shape[0],), fill_value = expert_idx, device = sample.device),
            )
            normal_log_prob = expert.log_prob(sample).sum(dim = -1)
            log_probs.append(expert_log_prob + normal_log_prob)
        log_probs = torch.stack(log_probs, dim = 1)
        return torch.logsumexp(log_probs, dim = 1)

    def sample(self) -> Tensor:
        with torch.no_grad():
            expert_samples = []
            for expert in self._experts:
                expert_samples.append(expert.sample())
            expert_samples = torch.stack(expert_samples, dim = 1)
            expert_idx = self._selector.sample().view(-1, 1, 1)
            expert_idx = expert_idx.expand(-1, 1, expert_samples.shape[2])
            chosen_sample = expert_samples.gather(dim = 1, index = expert_idx).squeeze(1)
            return chosen_sample
        
    def rsample(self) -> Tensor:
        expert_samples = []
        for expert in self._experts:
            expert_samples.append(expert.rsample())
        expert_samples = torch.stack(expert_samples, dim = 1)
        expert_idx = self._selector.sample().view(-1, 1, 1)
        expert_idx = expert_idx.expand(-1, 1, expert_samples.shape[2])
        chosen_sample = expert_samples.gather(dim = 1, index = expert_idx).squeeze(1)
        return chosen_sample

    def rsample_soft(self) -> tuple[Tensor, Tensor]:
        expert_samples = []
        for expert in self._experts:
            expert_samples.append(expert.rsample())
        expert_samples = torch.stack(expert_samples, dim = 1)
        expert_weights = self._selector.probs
        return expert_samples, expert_weights
    
    def negative_entropy_estimate(self) -> Tensor:
        return self.log_prob(self.sample())
    
    @staticmethod
    def diagonal_gaussian_kl(mu1: Tensor, sigma1: Tensor, mu2: Tensor, sigma2: Tensor) -> Tensor:
        var1 = sigma1 ** 2
        var2 = sigma2 ** 2
        return .5 * torch.sum(
            var1 / var2 + (mu2 - mu1) ** 2 / var2 - 1 + torch.log(var2 / var1),
            dim = -1,
        )

    def expert_diversity(self) -> Tensor:
        probs = self._selector.probs

        kl_sum = 0
        num_experts = len(self._experts)

        for i in range(num_experts):
            mu1 = self._mus[:, i]
            sigma1 = self._sigmas[:, i]
            p1 = probs[:, i].unsqueeze(-1)
            for j in range(i + 1, num_experts):
                mu2 = self._mus[:, j]
                sigma2 = self._sigmas[:, j]
                p2 = probs[:, j].unsqueeze(-1)
                kl_ij = self.diagonal_gaussian_kl(mu1, sigma1, mu2, sigma2).clamp(max = max_kl)
                kl_ji = self.diagonal_gaussian_kl(mu2, sigma2, mu1, sigma1).clamp(max = max_kl)
                weight = p1 * p2
                kl_sum = kl_sum + weight * (kl_ij + kl_ji)
        denom = num_experts * (num_experts - 1)
        return kl_sum / denom

policy = DiagonalGaussianMixture(mus=torch.ones(64, 10, 2), sigmas=torch.ones(64, 10, 2), logits=torch.ones(64, 10, requires_grad=True))
sample = policy.sample()
samples, weights = policy.rsample_soft()
assert weights.grad_fn is not None

In [ ]:
alpha = 0.002
diversity_strength = 0.1

In [126]:
from tqdm import trange
from torch.distributions import MultivariateNormal
from torch.nn.functional import mse_loss
from torch.nn.utils import clip_grad_norm_

num_episodes = 1000000
batch_size = 128
replay_capacity = 20000
gamma = 0.99
actor_lr = 3e-4
critic_lr = 3e-4
actor_polyak = 0.005
critic_polyak = 0.005
actor_update_interval = 1
update_every_steps = 20
max_grad_norm = 0.5
alpha_decay = 0.9995
alpha_end = 0.0001
action_scale = 20
diversity_decay = 0.9999
max_diversity = 0.4

actor_optimizer = optim.Adam(actor.parameters(), lr = actor_lr)
critic1_optimizer = optim.Adam(critic1.parameters(), lr = critic_lr)
critic2_optimizer = optim.Adam(critic2.parameters(), lr = critic_lr)

replay_buffer = ReplayBuffer(
    capacity = replay_capacity,
    batch_size = batch_size,
    device = device,
    rng = rng,
)

def get_multivariate_normal(params):
    loc = params[:, : action_dim]
    scale = params[:, action_dim :]
    L = torch.zeros(params.shape[0], action_dim, action_dim, device = device)
    tril_indices = torch.tril_indices(row = action_dim, col = action_dim)
    L[:, tril_indices[0], tril_indices[1]] = scale
    L_diag = L.diagonal(dim1 = -2, dim2 = -1)
    L_diag[:] = torch.exp(L_diag).clamp(min = 1e-3)
    return MultivariateNormal(loc, scale_tril = L)

iter_count = 0
update_count = 0
episode_rewards = []
for episode in trange(num_episodes):
    done = False
    episode_reward = 0
    state, _ = env.reset()

    while not done:
        with torch.no_grad():
            state = torch.tensor(state, dtype = torch.float32, device = device).unsqueeze(0)
            policy_params = actor(state)
            policy = DiagonalGaussianMixture.from_params(policy_params, num_experts, action_dim)
            action = policy.sample()

        next_state, reward, terminated, truncated, _ = env.step(action[0].cpu().numpy() / action_scale)
        episode_reward += reward
        done = terminated or truncated
        replay_buffer.add((
            state,
            action,
            torch.tensor([[reward]], dtype = torch.float32),
            torch.tensor([[done]], dtype = torch.int32),
            torch.tensor([next_state], dtype = torch.float32),
        ))

        if iter_count % update_every_steps == 0 and replay_buffer.ready():
            batch_state, batch_action, batch_reward, batch_done, batch_next_state = replay_buffer.sample()

            critic1_optimizer.zero_grad(set_to_none = True)
            critic2_optimizer.zero_grad(set_to_none = True)
            with torch.no_grad():
                next_policy_params = actor(batch_next_state)
                next_policy = DiagonalGaussianMixture.from_params(next_policy_params, num_experts, action_dim)
                next_action = next_policy.sample()
                next_log_prob = next_policy.log_prob(next_action).unsqueeze(1)
                next_q1 = target_critic1(torch.cat([batch_next_state, next_action], dim = 1))
                next_q2 = target_critic2(torch.cat([batch_next_state, next_action], dim = 1))
                next_q = torch.min(next_q1, next_q2)
                target_q = batch_reward + gamma * (1 - batch_done) * (next_q - alpha * next_log_prob)
            q = critic1(torch.cat([batch_state, batch_action], dim = 1))
            critic1_loss = mse_loss(q, target_q)
            critic1_loss.backward()
            clip_grad_norm_(critic1.parameters(), max_grad_norm)
            critic1_optimizer.step()
            polyak_update(target_critic1, critic1, critic_polyak)
            q = critic2(torch.cat([batch_state, batch_action], dim = 1))
            critic2_loss = mse_loss(q, target_q)
            critic2_loss.backward()
            clip_grad_norm_(critic2.parameters(), max_grad_norm)
            critic2_optimizer.step()
            polyak_update(target_critic2, critic2, critic_polyak)
            
            actor_optimizer.zero_grad(set_to_none = True)
            policy_params = actor(batch_state)
            policy = DiagonalGaussianMixture.from_params(policy_params, num_experts, action_dim)
            expert_actions, expert_weights = policy.rsample_soft()
            assert expert_weights.grad_fn is not None
            batch_state_expanded = batch_state.unsqueeze(1).expand(-1, num_experts, -1)
            q1 = target_critic1(torch.cat([batch_state_expanded, expert_actions], dim = 2))
            q2 = target_critic2(torch.cat([batch_state_expanded, expert_actions], dim = 2))
            q = (torch.min(q1, q2).squeeze(2) * expert_weights).mean(dim = 1)
            negative_entropy_estimate = policy.negative_entropy_estimate()
            actor_loss = (alpha * negative_entropy_estimate - q).mean()
            diversity = policy.expert_diversity()
            actor_loss = actor_loss - diversity.clamp(max = max_diversity).mean()
            actor_loss.backward()
            clip_grad_norm_(actor.parameters(), max_grad_norm)
            actor_optimizer.step()
            polyak_update(target_actor, actor, actor_polyak)

            if update_count % 100 == 0:
                print(f"{update_count}, al: {actor_loss.item():.8f}, cl: {critic1_loss.item():.8f}, er: {episode_rewards[-1] if episode_rewards else 0:.8f}, a: {diversity_strength:.8f}, d: {diversity.min().item():.8f}, p: {policy._selector.probs.detach().max().item():.8f}")
            update_count += 1
            alpha = max(alpha * alpha_decay, alpha_end)
            diversity_strength *= diversity_decay
        state = next_state
        iter_count += 1
    
    episode_rewards.append(episode_reward)

  0%|          | 0/1000000 [00:00<?, ?it/s]

0, al: -1.99595213, cl: 0.17260343, er: 0.00000000, a: 0.08925206, d: 0.40647051, p: 0.57892507


  0%|          | 2/1000000 [00:12<1752:35:55,  6.31s/it]

100, al: -2.16104126, cl: 0.08961410, er: 315.56723252, a: 0.08836394, d: 0.39671183, p: 0.61076587


  0%|          | 4/1000000 [00:26<1850:50:35,  6.66s/it]

200, al: -2.33576941, cl: 0.10426819, er: 547.78335920, a: 0.08748466, d: 0.39579278, p: 0.61191267


  0%|          | 6/1000000 [00:39<1820:45:36,  6.55s/it]

300, al: -2.44817972, cl: 0.11423879, er: 809.30644246, a: 0.08661413, d: 0.40100372, p: 0.59696084


  0%|          | 6/1000000 [00:42<1985:43:21,  7.15s/it]


KeyboardInterrupt: 

In [ ]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import torch

# 1. Setup the Env with Recording
# Use 'rgb_array' so the wrapper can capture the frames
env = gym.make("HalfCheetah-v5", render_mode="rgb_array")

# Wrap it to save to a specific folder
env_recording = RecordVideo(
    env, 
    video_folder="./target_actor_results",
    episode_trigger=lambda x: True, # Record every episode in this run
    name_prefix="final_eval"
)

def record_stochastic_target_actor(num_episodes=5):    
    for ep in range(num_episodes):
        state, _ = env_recording.reset()
        done = False
        truncated = False
        ep_reward = 0
        
        while not (done or truncated):
            with torch.no_grad():
                # 1. Prepare state tensor
                state_t = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
                
                # 2. Get distribution params from the Target Actor
                policy_params = actor(state_t)
                
                # 3. Use your training helper to build the distribution
                policy = DiagonalGaussianMixture.from_params(policy_params, num_experts, action_dim)
                
                # 4. STOCHASTIC: Sample instead of using the mean
                # We use .sample() here because we don't need gradients for eval
                action = policy.sample() 
                
                # 5. Apply your specific scaling (c=2, s=7)
                action = action / action_scale
            
            # Step the environment
            state, reward, done, truncated, _ = env_recording.step(action[0].cpu().numpy())
            ep_reward += reward
            
        print(f"Stochastic Episode {ep} Finished. Reward: {ep_reward:.2f}")

    env_recording.close()

# Run the stochastic evaluation
record_stochastic_target_actor()

c:\Users\abhay\anaconda3\envs\ursa25\Lib\site-packages\gymnasium\wrappers\rendering.py:293: UserWarning: WARN: Overwriting existing videos at c:\Users\abhay\Downloads\mutualistic-agents-e2268e29df84f0af8bb2ef4a263a9ddd60d5ce75\mutualistic-agents-e2268e29df84f0af8bb2ef4a263a9ddd60d5ce75\tests\target_actor_results folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Stochastic Episode 0 Finished. Reward: 509.87
Stochastic Episode 1 Finished. Reward: 594.01
Stochastic Episode 2 Finished. Reward: 603.47
Stochastic Episode 3 Finished. Reward: 498.93
